In [24]:
!pip install -q beautifulsoup4 requests groq fuzzywuzzy python-Levenshtein lxml

In [ ]:
# ================================
# 🏆 Hackathon Template Notebook
# Prospect Research Agent
# ================================
import requests
import json
import re
import time
import traceback
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from groq import Groq


_API_KEY = "GROQ_API_KEY"





In [26]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

# Probed in order; stops once max_extra are confirmed live
PRIORITY_PATHS = [
    "/about", "/about-us", "/company", "/who-we-are",
    "/contact", "/contact-us", "/get-in-touch",
    "/services", "/solutions", "/what-we-do", "/products",
    "/team", "/our-team", "/people",
]

In [27]:
def normalize_url(url: str) -> str:
    url = url.strip().rstrip("/")
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    return url

def base_of(url: str) -> str:
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}"

def probe_url(url: str, timeout: int = 7) -> bool:
    """
    FIX 1: Use GET with stream=True instead of HEAD.
    HEAD is blocked by Cloudflare, Vercel, and most CDNs (405/403).
    stream=True means we only download headers — same speed, far higher success rate.
    """
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout,
                         allow_redirects=True, stream=True)
        r.close()
        return r.status_code == 200
    except Exception:
        return False

def discover_pages(root: str, max_extra: int = 4) -> list:
    """
    Probe PRIORITY_PATHS and return homepage + up to max_extra live pages.
    No sitemap, no fuzzy matching — deterministic and reliable.
    """
    pages = [root]
    for path in PRIORITY_PATHS:
        if len(pages) - 1 >= max_extra:
            break
        url = root + path
        if probe_url(url):
            pages.append(url)
            print(f"       ✓ {path}")
    return pages

In [28]:
# FIX 2: Detect Cloudflare/bot-wall pages that return 200 but contain no real content
_BOT_SIGNALS = [
    "just a moment", "checking your browser", "enable javascript",
    "cf-browser-verification", "captcha", "access denied",
    "ray id", "please wait", "ddos protection",
]

def is_bot_blocked(html: str) -> bool:
    """Returns True if the page is a JS-challenge / bot wall."""
    sample = html[:3000].lower()
    return any(sig in sample for sig in _BOT_SIGNALS)

def fetch_html(url: str) -> str | None:
    """Try 2 user-agents; returns raw HTML or None. Rejects bot-wall pages."""
    uas = [
        HEADERS["User-Agent"],
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/605.1.15 Safari/605.1.15",
    ]
    for ua in uas:
        try:
            r = requests.get(url, headers={**HEADERS, "User-Agent": ua},
                             timeout=15, allow_redirects=True)
            if r.status_code == 200 and not is_bot_blocked(r.text):
                return r.text
        except Exception:
            continue
    return None

def jina_fetch(url: str) -> str | None:
    """
    Jina AI Reader — renders JS-heavy SPAs server-side, returns clean text.
    Free, no API key needed. Used as fallback when requests returns sparse/blocked content.
    """
    try:
        r = requests.get(
            f"https://r.jina.ai/{url}",
            headers={"Accept": "text/plain", "X-Return-Format": "text"},
            timeout=30,
        )
        if r.status_code == 200 and len(r.text.strip()) > 200:
            return r.text.strip()
    except Exception:
        pass
    return None


In [29]:
_NOISE_TAGS = ["script", "style", "noscript", "iframe", "svg",
               "img", "meta", "link", "button", "form", "nav", "footer"]
_NOISE_CLASSES = re.compile(
    r"cookie|banner|popup|modal|advertisement|newsletter|"
    r"breadcrumb|pagination|social|subscribe|gdpr|consent|promo",
    re.I,
)

def clean_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    # FIX 10: wrap in list() so decomposing a parent doesn't skip siblings in the iterator
    for tag in list(soup(_NOISE_TAGS)):
        tag.decompose()
    for tag in list(soup.find_all(True)):
        try:
            cls = " ".join(tag.get("class") or [])
            tid = str(tag.get("id") or "")
            if _NOISE_CLASSES.search(cls) or _NOISE_CLASSES.search(tid):
                tag.decompose()
        except Exception:
            continue
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s{2,}", " ", text).strip()

def chunk_for_llm(text: str, max_chars: int = 4500) -> str:
    """Head-heavy split: company info and CTAs cluster at top and bottom of pages."""
    if len(text) <= max_chars:
        return text
    head = int(max_chars * 0.6)
    return text[:head] + "\n\n[...truncated...]\n\n" + text[-(max_chars - head):]


In [30]:
_EMAIL_RE = re.compile(
    r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,6}(?!\.\w)"
)
_BAD_EMAIL_DOMAINS = {
    "example.com", "sentry.io", "wixpress.com", "squarespace.com",
    "wordpress.com", "yourdomain.com", "email.com", "domain.com",
    "2x.png", "2x.jpg", "2x.webp",
}
_BAD_EMAIL_EXTS  = re.compile(r"\.(png|jpg|jpeg|gif|svg|webp|css|js|json|2x)$", re.I)
# FIX 6: filter obvious no-reply / system addresses that aren't real contacts
_BAD_EMAIL_LOCAL = re.compile(
    r"^(noreply|no-reply|donotreply|do-not-reply|mailer-daemon|postmaster|bounce|"
    r"unsubscribe|notifications?|alerts?|support-noreply|auto-?reply)$", re.I
)

def extract_emails(text: str) -> list:
    found = _EMAIL_RE.findall(text)
    out = []
    for e in found:
        local, domain = e.rsplit("@", 1)
        domain = domain.lower()
        if domain in _BAD_EMAIL_DOMAINS:
            continue
        if _BAD_EMAIL_EXTS.search(domain):
            continue
        if _BAD_EMAIL_LOCAL.match(local):        # FIX 6
            continue
        out.append(e.lower())
    return list(dict.fromkeys(out))

# FIX 5: Tighter phone regex — requires proper 3-digit area code + 7-digit local number
# Uses word boundaries to avoid matching version strings (3.3.70), dates (2024-01-15), etc.
_PHONE_RE = re.compile(
    r"(?<!\d)"                          # not preceded by a digit
    r"(\+?1[\s\-.]?)?"                  # optional +1 country code
    r"(\(?\d{3}\)?[\s\-.])"            # area code (with or without parens)
    r"(\d{3}[\s\-.])"                   # exchange
    r"(\d{4})"                          # subscriber number
    r"(?!\d)"                           # not followed by a digit
)

def extract_phones(text: str) -> list:
    found = _PHONE_RE.findall(text)
    out = []
    for groups in found:
        p = "".join(groups).strip()
        digits = re.sub(r"\D", "", p)
        if 10 <= len(digits) <= 15:     # enforce ≥10 digits to reject short noise
            out.append(p)
    return list(dict.fromkeys(out))


In [31]:

def ai_extract(text_blob: str, url: str, emails: list, phones: list) -> dict:
    client = Groq(api_key=API_KEY)  # FIX 3: Groq already imported at top

    contact_hint = ""
    if emails:
        contact_hint += f"\n[EMAILS FOUND ON PAGE — use exactly these]: {', '.join(emails[:5])}"
    if phones:
        contact_hint += f"\n[PHONES FOUND ON PAGE — use first valid one]: {', '.join(phones[:3])}"

    prompt = f"""You are a business intelligence analyst. Extract structured data from the website content below.

URL: {url}{contact_hint}

CONTENT:
{text_blob}

RULES:
- Only use information explicitly present in the content. Never invent contact details or services.
- If a field is missing, return "" for strings or [] for arrays — never null.
- "probable_pain_point": one logical inference from their service is acceptable here.
- "outreach_opener": must reference something specific from their actual business.
- Return ONLY a raw JSON object. No markdown fences, no preamble, no explanation.

OUTPUT FORMAT:
{{
  "website_name": "Brand name shown on site",
  "company_name": "Full legal name if available, else brand name",
  "address": "Physical address or N/A",
  "mobile_number": "Phone number or N/A",
  "mail": ["list", "of", "real", "emails"],
  "core_service": "One sentence — what they do",
  "target_customer": "Who they serve",
  "probable_pain_point": "Key challenge their customers face",
  "outreach_opener": "Personalized 1-2 sentence cold email opener"
}}"""

    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=900,
    )

    content = resp.choices[0].message.content
    if not content:
        raise ValueError("LLM returned empty content")

    # Strip markdown fences if present
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", content.strip(), flags=re.MULTILINE).strip()

    # FIX 7: Fallback JSON extraction if LLM wrapped the object in extra text
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
        else:
            raise ValueError(f"Cannot parse JSON from LLM response:\n{raw[:300]}")

    if not isinstance(parsed, dict):
        raise ValueError(f"LLM returned {type(parsed).__name__}, expected dict")

    return parsed

In [32]:
def enrich_company(url: str) -> dict:
    """
    Input:  Company URL (string)
    Output: Structured company profile dict with exactly 9 keys
    """
    DEFAULT = {
        "website_name": "N/A", "company_name": "N/A",
        "address": "N/A",      "mobile_number": "N/A",
        "mail": [],            "core_service": "N/A",
        "target_customer": "N/A", "probable_pain_point": "N/A",
        "outreach_opener": "N/A",
    }

    try:
        url  = normalize_url(url)
        root = base_of(url)

        # ── Step 1: Discover live pages ──────────────────────
        print(f"\n  [1/4] Probing priority paths on {root} ...")
        pages = discover_pages(root, max_extra=4)
        paths = [urlparse(p).path or "/" for p in pages]
        print(f"  [2/4] Scraping {len(pages)} page(s): {paths}")

        # ── Step 2: Scrape ───────────────────────────────────
        combined_text = ""
        all_emails: list = []
        all_phones: list = []

        for page_url in pages:
            time.sleep(0.4)
            html = fetch_html(page_url)

            # Extract emails from raw HTML — catches mailto: and data-email attributes
            if html:
                all_emails.extend(extract_emails(html))

            page_text = clean_html(html) if html else ""

            # Fallback to Jina for sparse / JS-blocked / bot-walled pages
            if len(page_text.split()) < 80:
                print(f"       ↳ Sparse ({len(page_text.split())} words) — Jina Reader fallback...")
                jina_text = jina_fetch(page_url)
                if jina_text:
                    page_text = jina_text
                    # FIX 4: also extract emails from Jina's plain-text output
                    all_emails.extend(extract_emails(jina_text))

            if not page_text:
                continue

            all_phones.extend(extract_phones(page_text))
            combined_text += f"\n\n=== {urlparse(page_url).path or '/'} ===\n{page_text[:1800]}"

        if not combined_text.strip():
            print("  ✗  No readable content found.")
            return {**DEFAULT, "website_name": urlparse(root).netloc}

        # ── Step 3: LLM enrichment ───────────────────────────
        optimized     = chunk_for_llm(combined_text)
        unique_emails = list(dict.fromkeys(all_emails))[:5]
        unique_phones = list(dict.fromkeys(all_phones))[:3]

        print(f"  [3/4] Calling LLM — ~{len(optimized) // 4} tokens | "
              f"{len(unique_emails)} email(s) | {len(unique_phones)} phone(s) pre-extracted ...")

        result = ai_extract(optimized, url, unique_emails, unique_phones)

        # ── Step 4: Harden contacts (regex beats LLM for accuracy) ──
        if unique_emails:
            result["mail"] = unique_emails
        if unique_phones and result.get("mobile_number", "N/A") in ("N/A", "", None):
            result["mobile_number"] = unique_phones[0]

        # ── Step 5: Schema guard ─────────────────────────────
        for k, v in DEFAULT.items():
            # FIX 8: treat empty string the same as None — fill with default
            if k not in result or result[k] is None or result[k] == "":
                result[k] = v
        if isinstance(result["mail"], str):
            result["mail"] = [result["mail"]] if result["mail"] else []

        print(f"  [4/4] ✓  {result.get('company_name', 'Unknown')} — done")
        return result

    except Exception as exc:
        # FIX 9: full traceback so root cause is visible during debugging
        print(f"  ✗  Error: {exc}")
        traceback.print_exc()
        return {**DEFAULT, "website_name": urlparse(normalize_url(url)).netloc}



In [33]:
EXAMPLE_URLS = [
    "https://www.atlassian.com",   # B2B SaaS — clean static pages
    "https://basecamp.com",        # simple static site
    "https://www.intercom.com",    # B2B messaging / CRM
    "https://www.zendesk.com",     # customer support SaaS
    "https://www.freshdesk.com",   # helpdesk SaaS
]

if __name__ == "__main__":
    print("=" * 60)
    print("🏢 Prospect Research Agent — Batch Enrichment")
    print("=" * 60)
    print("Pre-loaded examples:")
    for i, u in enumerate(EXAMPLE_URLS, 1):
        print(f"  {i}. {u}")
    print()
    print("Paste your own JSON array and press Enter to override,")
    print("or just press Enter to run the examples above.")
    print()

    raw_input = input("URLs (JSON array or blank for examples): ").strip()

    if not raw_input:
        urls = EXAMPLE_URLS
        print(f"\n▶  Running {len(urls)} pre-loaded example(s).\n")
    else:
        try:
            urls = json.loads(raw_input)
            if isinstance(urls, str):
                urls = [urls]
        except json.JSONDecodeError:
            urls = [u.strip().strip("\"'") for u in raw_input.split(",") if u.strip()]

    if not urls:
        print("⚠️  No URLs found. Exiting.")
    else:
        print(f"\n📋 Processing {len(urls)} URL(s)...\n")
        results = []
        for idx, url in enumerate(urls, 1):
            print(f"── [{idx}/{len(urls)}] {url}")
            try:
                results.append(enrich_company(url))
            except Exception as e:
                print(f"  ✗  Unhandled: {e}")
                traceback.print_exc()

        with open("results.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print("\n✅ Saved → results.json")

        print("\n" + "=" * 60)
        print("=== FINAL OUTPUT ===")
        print("=" * 60)
        print(json.dumps(results, indent=2, ensure_ascii=False))

🏢 Prospect Research Agent — Batch Enrichment
Pre-loaded examples:
  1. https://www.atlassian.com
  2. https://basecamp.com
  3. https://www.intercom.com
  4. https://www.zendesk.com
  5. https://www.freshdesk.com

Paste your own JSON array and press Enter to override,
or just press Enter to run the examples above.

URLs (JSON array or blank for examples): 

▶  Running 5 pre-loaded example(s).


📋 Processing 5 URL(s)...

── [1/5] https://www.atlassian.com

  [1/4] Probing priority paths on https://www.atlassian.com ...
       ✓ /company
       ✓ /contact
       ✓ /solutions
       ✓ /products
  [2/4] Scraping 5 page(s): ['/', '/company', '/contact', '/solutions', '/products']
  [3/4] Calling LLM — ~1130 tokens | 1 email(s) | 3 phone(s) pre-extracted ...
  [4/4] ✓  Atlassian — done
── [2/5] https://basecamp.com

  [1/4] Probing priority paths on https://basecamp.com ...
       ✓ /about
       ✓ /contact
       ✓ /contact-us
       ✓ /team
  [2/4] Scraping 5 page(s): ['/', '/about', '/con